# Python Setup

In [1]:
from sympy import *
from IPython.display import display, Math, Latex, Markdown
#init_session()
init_printing(use_unicode=True)

#init_printing(latex_printer=lambda *args, **kwargs: latex(*args, mul_symbol='dot', **kwargs))
def md(s):
    display(Markdown(s))
    
def note(s):
    md(f'<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">{s}</span></div>')

In [2]:
import re

class dotdict(dict):
    __getattr__ = dict.get
    #__setattr__ = dict.__setitem__
    __delattr__ = dict.__delitem__
    
    def __setattr__(self, key, val):
        self.set(key, val)

    def set(self, key, val, extra_subs=None, unit=''):
        if not isinstance(val, str):
            return dict.__setitem__(self, key, val)
        if not unit:
            m = re.search(r'\[(.+?)\]', val)
            if m:
                unit = m.group(1)
                val = re.sub(r'\[(.+?)\]', '', val)
        ## Consider any string passe in as a SymPy expression
        expr = sympify(val, evaluate=False)

        with evaluate(False):
            expr_subs = expr.subs(self | (extra_subs or {}))
            subtituted_ltx = latex(expr_subs,mul_symbol='dot',order='lex')
        res = expr_subs.doit()
        eq = f'{key} = {latex(expr,mul_symbol='dot',order='lex')} = {subtituted_ltx} = {N(res, 5)} \\, {unit}'
        #print(eq)
        display(Math(eq))
        return dict.__setitem__(self, key, res)

# Geometrie

In [3]:
geom = dotdict({
    'd': 56000,
    'l_h1': 5000,
    'l_h2': 12500,
    'h_a': 6500,
    'stuetzweite': 6750,
})
geom.l_h = 'l_h1 + l_h2'

geom.α = atan(5/100)
geom.l = 'l_h/cos(α)'
geom.b = 'l'
geom.Δh = 'l_h * tan(α)'
geom.h = 'h_a + Δh'
geom.A = 'l * d'
#geom

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [28]:
#TODO choose material, rn I profil 15x15??

# HEB 240
obergurt = dotdict({
    'b': 300, #mm
    'd': 300, #mm
    
})

# HEB 240
untergurt = dotdict({
    'b': 300, #mm
    'd': 300, #mm
})

# HEA 140
streben = dotdict({
    'd': 160, #mm
})

# HEB 160
stuetzeA = dotdict({
    'l': 6500, # mm
    'b': 160, #mm
    'd': 160, #mm
    #'Iy': 2492e4, #mm^4
    #'Iz': 889.2e4, #mm^4
    #'A': 5425, #mm^2
})

# HEB 320
stuetzeB = dotdict({
    'l': 7750, # mm
    'b': 300, #mm
    'd': 650, #mm
   # 'Iy': 210600e4, #mm^4
   # 'Iz': 13980e4, #mm^4
   # 'A': 28600, # mm^2
})

binder = dotdict({
    'b': 300,
    'A_x': (22000 * streben.d) + (18000 * obergurt.d) + (17000 * untergurt.d),
    'h': 1500,
})
pfette = dotdict({
    'l': geom.stuetzweite,
    'e': geom.l/7,
    'abstand': geom.l/7,
    'stuetzweite': geom.stuetzweite,
    
})
pfette

{'l': 6750,
 'e': 2503.12304931260,
 'abstand': 2503.12304931260,
 'stuetzweite': 6750}

# Lastaufstellung

## Dachaufbau

In [5]:
dauerlasten = dotdict()
dauerlasten.set('g_wellblech', '0.2 * e/1000', pfette, unit='kN/m')
# 0.2 auf der sicheren seite, siehe Kataloge:
# https://www.proge.de/fileadmin/contents/Belastungstabellen/Trapezbleche_aus_Stahl/Belastugnstabelle-Trapezblech-Stahl-Profil-100-275-Positivlage.pdf
# https://www.proge.de/fileadmin/contents/Belastungstabellen/Trapezbleche_aus_Stahl/Belastungstabelle-Stahl-45-150-55-Positivlage.pdf
#
# Vorauswahl: Profil 85/280 t=1.5mm


<IPython.core.display.Math object>

## Pfette

In [6]:
#dauerlasten.set('g_pfette', '(7.75 + 0.12) * 0.423', pfette, unit='kN')
## Längste Pfette am Rand (mit kragarm): 7.75m
## Überlappung an linkem Binder: HEB240 * 0.5 => 0.12m
## last HEA 200: 0.423 kN/m lt https://www.ezzat.org/de/Querschnittswerte/gewalzt/I/hea.php

## ignore: wird von RFEM beruecksichtig!!

## Schnee

In [7]:
schnee = dotdict({ 'C_e': 1, 'C_t':  1, 'mu_i':  0.8, 's_k':  3.2, })

schnee.s = 'C_e * C_t * mu_i * s_k [kN/m^2]'
md('#### ✅ [11] Schneelast')
schnee.set('gleichlast', 's * (e/1000) [kN/m]', pfette)
schnee.set('feldkraft', 'gleichlast * stuetzweite/1000 [kN]', geom)

<IPython.core.display.Math object>

#### ✅ [11] Schneelast

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Nutzlast

In [8]:
print(f'Nutzlast g_k = {(pfette.e/1000):0.4} kN/m')

Nutzlast g_k = 2.503 kN/m


## Windlast

In [9]:
def init_wind():
    ## Werte lt hora.gv.at für "Bad Ischl (Stadt)"
    state = dotdict({
        'q_b0': 0.29,  # [kN/m^2]
        'v_b0': 21.50,  # [m/s] 
        'z_e': geom.h,  # [m]
        'cf_Binder': 2, #lt Angabe
    })
    
    ## Gelaendekategorie II
    state.q_p = 'q_b0 * 2.1 * ((z_e * 1e-3)/10)**0.24'
    return state
    

In [10]:
wind_nord = init_wind()
wind_nord

<IPython.core.display.Math object>

{'q_b0': 0.29,
 'v_b0': 21.5,
 'z_e': 7375.00000000000,
 'cf_Binder': 2,
 'q_p': 0.566083085684517}

## Wind ➡️ OST (0°)

In [11]:
wind_ost = init_wind()
wind_ost.set('φ', 'Δh/h', geom)
wind_ost

<IPython.core.display.Math object>

<IPython.core.display.Math object>

{'q_b0': 0.29,
 'v_b0': 21.5,
 'z_e': 7375.00000000000,
 'cf_Binder': 2,
 'q_p': 0.566083085684517,
 'φ': 0.118644067796610}

Es wirken:

 - Winddruck auf Dachfläche $w_k$
 - Reibung auf Dachfläche $w_r$
 - Winddruck auf Stütze (Reihe A)
 - Winddruck auf Wandriegel

In [12]:
def interp_alpha(val0, val5):
    return N(val0 + deg(geom.α) * ((val5-val0)/5),2)

#### Winddruck auf Dachfläche

In [13]:

def interp_phi(val0, val1):
    return N(val0 + wind_ost.φ * (val1-val0),2)
    
## Flächeneinteilung 
md('**Dachflächen**')
geom.Dach_A = '(d - 2 * d/10) * (b - 2 * b/10) * 1e-6'
geom.Dach_B = '2 * (d * b/10) * 1e-6'
geom.Dach_C = '2 * (d/10 * b) * 1e-6'

# φ = 0 von dieser Richtung
#cf_max = interp_alpha(0.2, 0.4)
#cf_min = interp_alpha(-0.5, -0.7)
cpnet_maxA = interp_alpha(0.5, 0.8)
cpnet_maxB = interp_alpha(1.8, 2.1)
cpnet_maxC = interp_alpha(1.1, 1.3)

cpnet_minA = interp_alpha(interp_phi(-0.6, -1.5), interp_phi(-1.1, -1.6))
cpnet_minB = interp_alpha(interp_phi(-1.3, -1.8), interp_phi(-1.7, -2.2))
cpnet_minC = interp_alpha(interp_phi(-1.4, -2.2), interp_phi(-1.8, -2.5))

#print(cpnet_maxA)
#print(cpnet_maxB)
#print(cpnet_maxC)
#
#print(geom.Dach_A/(geom.A * 1e-6))
#print(geom.Dach_B/(geom.A * 1e-6))
#print(geom.Dach_C/(geom.A * 1e-6))

wind_ost.cpnet_min = sum([(geom.Dach_A/(geom.A * 1e-6)) * cpnet_minA,
                          (geom.Dach_B/(geom.A * 1e-6)) * cpnet_minB,
                          (geom.Dach_C/(geom.A * 1e-6)) * cpnet_minC])
wind_ost.cpnet_max = sum([(geom.Dach_A/(geom.A * 1e-6)) * cpnet_maxA,
                          (geom.Dach_B/(geom.A * 1e-6)) * cpnet_maxB,
                          (geom.Dach_C/(geom.A * 1e-6)) * cpnet_maxC])

#print(wind_ost.cpnet_min)
#print(wind_ost.cpnet_max)

md('### ✅ [21] Druck')
wind_ost.set('wk_1', 'q_p * cpnet_max * e/1000', pfette, unit='kN/m')

md('### ✅ [22] Sog')
wind_ost.set('wk_2', 'q_p * cpnet_min * e/1000', pfette, unit='kN/m')

**Dachflächen**

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### ✅ [21] Druck

<IPython.core.display.Math object>

### ✅ [22] Sog

<IPython.core.display.Math object>

### ✅ [23] Reibungskraft auf Dachfläche

In [14]:
# Da Trapezblech entlang windrichtung orientiert, kann ein $c_{fr} = 0.01$ angenommen werden
wind_ost.c_fr = 0.01
wind_ost.set('A_fr', '2 * l * d * 1e-6', geom, unit='m^2')

wind_ost.w_fr = 'c_fr * q_p'
wind_ost.F_fr = 'c_fr * q_p * A_fr'

#randbinder:
wind_ost.set('F_frBinder', 'w_fr * l/1000 * stuetzweite/1000', geom, unit='kN')
wind_ost.F_fr * geom.stuetzweite


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### ✅ [24] Windkraft auf Stütze (Reihe A)

In [15]:
## Annahme: Im bereich C der kurve
#wind_ost.set('cf0_col', '-0.721 * ln(d/b) + 2.1460', stuetzeA)
#wind_ost.ψ_γ = 1 # kantiger QS
#stuetzeA.λ = 'max(l/b, 70)'
#note('effektive schlankheit passt so? [EC 1-4, Tab 7.16]')
#wind_ost.ψ_λ = 1 #fuer φ = 0 und λ = 70 [EC 1-4, Abb. 7.36]
#wind_ost.cf_col = 'cf0_col * ψ_λ'

## NB: Stuetzen kraft wird nur zu 2/3 eingeleitet, rest geht ins fundament
wind_ost.set('F_frStuetze', 'q_p * cf_Binder * ((b*l)/1e6)', stuetzeA, unit='kN')



<IPython.core.display.Math object>

### ✅ [25] Windkraft auf Binder

In [16]:
wind_ost.set('F_wUG1', 'q_p * cf_Binder * (1484 * b)/1000**2', binder)

wind_ost.set('F_wUG2', 'q_p * cf_Binder * ((2400 + 1381) * b)/1000**2', binder)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Wind ⬅️ WEST (180°)

Es wirken:

 - Winddruck auf Dachfläche $w_k$
 - Reibung auf Dachfläche $w_r$
 - Winddruck auf Stütze (Reihe A)
 - Winddruck auf Wandriegel

In [17]:
wind_west = init_wind()
wind_west.set('φ', '(6000+Δh)/(h+6000)', geom)
wind_west

<IPython.core.display.Math object>

<IPython.core.display.Math object>

{'q_b0': 0.29,
 'v_b0': 21.5,
 'z_e': 7375.00000000000,
 'cf_Binder': 2,
 'q_p': 0.566083085684517,
 'φ': 0.514018691588785}

#### Winddruck auf Dachfläche

In [18]:
##### φ = elem (0,1) von dieser Richtung
def interp_phi(val0, val1):
    return N(val0 + wind_west.φ * (val1-val0),2)
    
cpnet_maxA = interp_alpha(0.5, 0.8)
cpnet_maxB = interp_alpha(1.8, 2.1)
cpnet_maxC = interp_alpha(1.1, 1.3)

print(cpnet_minA)
print(cpnet_minB)
print(cpnet_minC)

cpnet_minA = interp_alpha(interp_phi(-0.6, -1.5), interp_phi(-1.1, -1.6))
cpnet_minB = interp_alpha(interp_phi(-1.3, -1.8), interp_phi(-1.7, -2.2))
cpnet_minC = interp_alpha(interp_phi(-1.4, -2.2), interp_phi(-1.8, -2.5))

wind_west.cpnet_min = sum([(geom.Dach_A/(geom.A * 1e-6)) * cpnet_minA,
                           (geom.Dach_B/(geom.A * 1e-6)) * cpnet_minB,
                           (geom.Dach_C/(geom.A * 1e-6)) * cpnet_minC])
wind_west.cpnet_max = sum([(geom.Dach_A/(geom.A * 1e-6)) * cpnet_maxA,
                           (geom.Dach_B/(geom.A * 1e-6)) * cpnet_maxB,
                           (geom.Dach_C/(geom.A * 1e-6)) * cpnet_maxC])

print(wind_west.cpnet_min)
print(wind_west.cpnet_max)

md('### ✅ [26] Druck')
wind_west.set('wk_1', 'q_p * cpnet_max * e/1000', pfette, unit='kN/m')

md('### ✅ [27] Sog')
wind_west.set('wk_2', 'q_p * cpnet_min * e/1000', pfette, unit='kN/m')

-0.97
-1.6
-1.7
-1.54687500000000
1.06750000000000


### ✅ [26] Druck

<IPython.core.display.Math object>

### ✅ [27] Sog

<IPython.core.display.Math object>

### ✅ [28] Reibungskraft auf Dachfläche

In [19]:
# Da Trapezblech entlang windrichtung orientiert, kann ein $c_{fr} = 0.01$ angenommen werden
wind_west.c_fr = 0.01
wind_west.set('A_fr', '2 * l * d * 1e-6', geom, unit='m^2')
wind_west.F_fr = 'c_fr * q_p * A_fr'


<IPython.core.display.Math object>

<IPython.core.display.Math object>

### ✅ [29] Windkraft auf Stütze (Reihe B)

In [20]:
## Annahme: Im bereich C der kurve
wind_west.set('cf0_col', '-0.721 * ln(d/b) + 2.1460', stuetzeB)
#wind_west.ψ_γ = 1 # kantiger QS
#stuetzeB.λ = 'max(l/b, 70)'
#note('effektive schlankheit passt so? [EC 1-4, Tab 7.16]')
#wind_west.ψ_λ = 1 #fuer φ = 0 und λ = 70 [EC 1-4, Abb. 7.36]
#wind_west.cf_col = 'cf0_col * ψ_λ'

# NB: nur 2/3 werden in dach-anschluss eingeleitet, rest ins fundament
wind_west.set('F_wStuetze', 'q_p * cf_Binder * ((b*l)/1e6)', stuetzeB, unit='kN')



<IPython.core.display.Math object>

<IPython.core.display.Math object>

### ✅ [30] Windkraft FW-Binder

In [21]:
wind_west.set('F_wUGli', 'q_p * cf_Binder * (11345 * b)/1000**2', untergurt, unit='kN')
wind_west.set('F_wUGre', 'q_p * cf_Binder * (1469 * b)/1000**2', untergurt, unit='kN')

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Wind ⬆️ NORD (90°)

In [29]:
## Reibung

wind_nord.c_fr = 0.04   # [EC3-1-4, Tab 7.10] "Sehr rauh" weil Trapezblech
wind_nord.set('A_fr', '2 * l * (d) * 1e-6', geom, unit='m^2')
wind_nord.set('A_frPfette', '2 * e * stuetzweite * 1e-6', pfette, unit='m^2')
md('### ✅ [31] Reibung auf Dach')
wind_nord.set('F_frNord', 'c_fr * q_p * A_fr', unit='kN')
wind_nord.set('F_frNordPfette', 'c_fr * q_p * A_frPfette', unit='kN')
#wind_nord.set('F_frNordProVerbandPfette', 'F_frNord * (1/4) * (1/7) * 0.5')

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### ✅ [31] Reibung auf Dach

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### ✅ [33] Windkraft auf FW Binder

In [31]:
### Binder:
wind_nord.set('Fw_binder', 'cf_Binder * q_p * (A_x) * 1e-6', binder, unit='kN')

pro_pfette_aussen = wind_nord.Fw_binder * (1/7 * 0.5)
md(f'Aufalgerkraft pro pfette je binderabschnitt: {pro_pfette_aussen:0.4}/{pro_pfette_aussen*2:0.4} kN')

<IPython.core.display.Math object>

Aufalgerkraft pro pfette je binderabschnitt: 1.134/2.268 kN

### ✅ [32] Windkraft auf Stützen

In [33]:
wind_nord.set('Fw_StuetzeA', 'cf_Binder * q_p * (d * l)/1000**2 * (3/3)', stuetzeA, unit='kN')

wind_nord.set('Fw_StuetzeB', 'cf_Binder * q_p * (d * l)/1000**2 * (3/3)', stuetzeB, unit='kN')


<IPython.core.display.Math object>

<IPython.core.display.Math object>

# Lastfälle

## Grundlastfälle

 - `GLF01` Eigengewicht Fachwerkträger & Stützen
 - `GLF02` Pfetten
 - `GLF03` Ausbaulasten (Trapezblech)
<hr style="width: 7rem; margin-left: 1rem;"/>
 - `GLF11` Schnee
<hr style="width: 7rem; margin-left: 1rem;"/>
 - `GLF21` Wind ➡️: Dach Druck
 - `GLF22` Wind ➡️: Dach Sog
 - `GLF23` Wind ➡️: Dach Reibung
 - `GLF24` Wind ➡️: Stuetzen
 - `GLF25` Wind ➡️: Wandriegel (??)
<hr style="width: 7rem; margin-left: 1rem;"/>
 - `GLF26` Wind ️️⬅️: Dach Druck
 - `GLF27` Wind ️️⬅️: Dach Sog
 - `GLF28` Wind ️️⬅️: Dach Reibung (??)
 - `GLF29` Wind ⬅️️: Stuetzen
 - `GLF30` Wind ⬅️: Wandriegel (??)
<hr style="width: 7rem; margin-left: 1rem;"/>
 - `GLF31` Wind ⬆️️️: Dach Reibung
 - `GLF32` Wind ⬆️️️: Stuetzen
 - `GLF33` Wind ⬆️️️: Binder
<hr style="width: 7rem; margin-left: 1rem;"/>
 - `GLF41` Nuztlast für Instandsetzungsarbeiten: Dach Kategorie H

## Lastfälle

| LF    | ELF | Bezeichnung | Formel |
|-------|-----|-------------|--------|
| `LF1` |`ELF10` | Eigengewichte | `GLF01`|
| `LF2` |`ELF11` | Pfetten & Dachaufbau | `GLF02 + GLF03`|
| `LF3` |`ELF20` | Schnee, veraenderlich | `GLF11`|
| `LF4` |`ELF40` | Wind ➡️ Druck | `GLF21 + GLF23 + GLF24 + GLF25`|
|       |`ELF41` | Wind ➡️ Sog   | `GLF22 + GLF23 + GLF24 + GLF25`|
|       |`ELF42` | Wind ⬅️ Druck | `GLF26 + GLF28 + GLF29 + GLF30`|
| `LF5` |`ELF43` | Wind ⬅️ Sog   | `GLF27 + GLF28 + GLF29 + GLF30`|
| `LF6` |`ELF44` | Wind ⬆️️       | `GLF31 + GLF32 + GLF33`        |
|       |`ELF50` | Nutzlast, veraenderlich | `GLF41`|

## Lastkombinationen

| LK   |  Bezeichnung | Formel |
|------|--------------|--------|
|`LK1` | Ständige                      | ` 1.5 · LF1` |
|`LK2` | Schnee leitend + Wind (0°)    | ` 1.5 · LF1 + 1.35 · LF2 + (1.35 · 0.9) LF3`|
|`LK3` | Schnee leitend + Wind (180°)  | ` 1.5 · LF1 + 1.35 · LF2 + (1.35 · 0.9) LF4`|
|`LK4` | Schnee leitend + Wind (90°)   | ` 1.5 · LF1 + 1.35 · LF2 + (1.35 · 0.9) LF5`|
|`LK5` | Wind (0°) leitend + Schnee    | ` ...`|